In [2]:
import warnings 
warnings.filterwarnings('ignore')
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
loder = PyPDFLoader('C:\\Users\\SAYAN METE\\OneDrive\\Documents\\Retrival Argumented Generation\\hybrid_RAG\\Virat_Kohli_Complete_With_Personal_Life.pdf')
pages = loder.load()


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter = RecursiveCharacterTextSplitter(chunk_size = 2000,chunk_overlap = 200)
text = spliter.split_documents(pages)

chunk = []
for i in text:
    chunk.append(i.page_content)
metadata = []
for i in text:
    metadata.append(i.metadata)

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path="./database-3")
collection = client.get_or_create_collection(name="Collection",embedding_function=embedding_function)

try:
    if collection.count() == 0:
        collection.add(
            documents=chunk,
            ids = [str(i) for i in range(len(chunk))],
            metadatas=metadata
        )

except Exception as e:
    print(str(e))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2942.45it/s]


In [4]:
from rank_bm25 import BM25Okapi

def token_create(i):
    i = i.lower()
    i = i.split()
    return i
token = [token_create(i) for i in chunk]
token_for_keyword_search = BM25Okapi(token)


In [5]:
def retrival(query:str)->str:
    query_lower_case = query.lower()

    response = collection.query(query_texts=[query_lower_case],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    thresold = 1.6
    near_chunk = []
    for i , j in zip(distance,document):
        if thresold>i:
            near_chunk.append(j)
    score = token_for_keyword_search.get_scores(token_create(query_lower_case))

    def near_index_find(score,k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index, key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in index_sorted[:k]]

    get_index = near_index_find(score,k = 10)
    index_to_chunk = []
    for i in get_index:
        index_to_chunk.append(chunk[i])

    rrf_item = {}

    for rank , doc in enumerate(near_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_to_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    top_related_doc = []
    for doc,_ in merge:
        top_related_doc.append(doc)
    if not top_related_doc:
        return "NOT RELATED CONTENT"
    return "\n\n".join(top_related_doc)


    

In [6]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api = os.getenv('GROQ_API_KEY') 
groq_llm_model = ChatGroq(model='openai/gpt-oss-120b',api_key=api)

question = input('ask your question about virat kholi??')
content = retrival(question)

prompt = """
You are a realiable ai assistent so provide user asking qustions based on only the local document 

content = {content}
qustion = {question}
"""
final_prompt = prompt.format(content=content,question=question)
print(groq_llm_model.invoke(final_prompt).content)


**Virat Kohli – Runs (as recorded in the local document, September 2026)**  

| Format | Matches | Runs | Source note |
|--------|---------|------|-------------|
| **Test** | 123 | **9,230** | “Legendary Test career release” – 123‑Test career, 9,230 runs |
| **ODI** | 314 | **14,941** | BCCI profile – 14,941 runs (average 58.59, 54 centuries) |
| **T20I** | 125 | **4,188** | BCCI profile – 4,188 runs (average 48.70, 1 century) |

### Combined career runs (approximate)

Adding the three formats together gives a **total of roughly 28,359 runs** in international cricket (9,230 + 14,941 + 4,188).  

> **Note:** These figures are drawn from the official BCCI and ICC sources cited in the document and are current up to September 2026. Active‑format statistics may change after publication.
